In [1]:
from transformers import BertForSequenceClassification, BertTokenizerFast

model_name = "asqar268008/mini-bert-mlm-nsp-model"

In [2]:
tokenizer = BertTokenizerFast.from_pretrained(
    model_name
)

model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels = 2
)

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: asqar268008/mini-bert-mlm-nsp-model
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
from datasets import load_dataset

# Using the namespace-qualified identifier to bypass the parser issue on cached older packages
imdb = load_dataset("stanfordnlp/imdb")

train_dataset = imdb["train"].shuffle(seed=42).select(range(5000))
test_dataset = imdb["test"].shuffle(seed=42).select(range(2000))

In [4]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

In [5]:
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [6]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [7]:
trainig_args = TrainingArguments(
    output_dir = "mini-bert-mlm-nsp-model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 5,
    weight_decay = 0.01,
    logging_steps = 100,
    fp16 = True
)

In [8]:
trainer = Trainer(
    model = model,
    args = trainig_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    compute_metrics = compute_metrics
)

In [9]:
trainer.train()

Step,Training Loss
100,0.690902
200,0.683898
300,0.682027
400,0.661403
500,0.654111
600,0.632859
700,0.608970
800,0.587517
900,0.580401
1000,0.551704


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1565, training_loss=0.5944777833006253, metrics={'train_runtime': 83.2585, 'train_samples_per_second': 300.27, 'train_steps_per_second': 18.797, 'total_flos': 41764492800000.0, 'train_loss': 0.5944777833006253, 'epoch': 5.0})

In [10]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
0.507797,0.616625,1565,0.677500


{'eval_loss': 0.6166254878044128, 'eval_accuracy': 0.6775}

In [11]:
import torch
import torch.nn.functional as F

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 256, padding_idx=0)
      (position_embeddings): Embedding(128, 256)
      (token_type_embeddings): Embedding(2, 256)
      (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=256, out_features=256, bias=True)
              (LayerNorm): LayerNorm((256,), eps=1e-12, e

In [15]:
def predict(text):
  inputs = tokenizer(
      text,
      return_tensors = "pt",
      padding = True,
      truncation = True,
      max_length = 128
  )
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = model(**inputs)

  probs = F.softmax(outputs.logits, dim=1)
  prediction = torch.argmax(probs, dim=1).item()

  label = "Positive" if prediction == 1 else "Negative"
  return {
      "Text": text,
      "Prediction": label,
      "Probability": probs[0][prediction].item()
  }

In [16]:
text = "This was the worst film I have ever seen in my life."
predict(text)

{'Text': 'This was the worst film I have ever seen in my life.',
 'Prediction': 'Negative',
 'Probability': 0.5010604858398438}

In [17]:
text = "This was the amazing film I have ever seen in my life."
predict(text)

{'Text': 'This was the amazing film I have ever seen in my life.',
 'Prediction': 'Positive',
 'Probability': 0.8929365277290344}